In [3]:
!ls /kaggle/input/datasets/omswarooptm/densevit-cifar-100

dense_cifar100_best.pt


In [4]:
# ============================================================
# CIFAR-100 Sparse + KD
# ============================================================

import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as transforms
from torchvision.models import vit_b_16
from torch.utils.data import DataLoader
import copy

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# -------------------------
# Data
# -------------------------
def get_cifar100(batch_size=32):

    transform = transforms.Compose([
        transforms.Resize(224),
        transforms.ToTensor(),
        transforms.Normalize((0.5,)*3, (0.5,)*3)
    ])

    trainset = torchvision.datasets.CIFAR100("./data", train=True, download=True, transform=transform)
    testset = torchvision.datasets.CIFAR100("./data", train=False, download=True, transform=transform)

    trainloader = DataLoader(trainset, batch_size=batch_size, shuffle=True, num_workers=2)
    testloader = DataLoader(testset, batch_size=batch_size, shuffle=False, num_workers=2)

    return trainloader, testloader

train_dl, test_dl = get_cifar100()

@torch.no_grad()
def evaluate(model):
    model.eval()
    correct, total = 0, 0
    for x, y in test_dl:
        x, y = x.to(device), y.to(device)
        out = model(x)
        pred = out.argmax(dim=1)
        correct += (pred == y).sum().item()
        total += y.size(0)
    return correct / total

# -------------------------
# Sparse Wrapper
# -------------------------
class SparseViT(nn.Module):
    def __init__(self, base_model, keep_ratio=0.5):
        super().__init__()
        self.base = base_model
        self.keep_ratio = keep_ratio

    def forward(self, x):
        x = self.base._process_input(x)
        B, N, C = x.shape

        cls_token = self.base.class_token.expand(B, -1, -1)
        x = torch.cat((cls_token, x), dim=1)
        x = x + self.base.encoder.pos_embedding

        scores = x.norm(dim=-1)
        k = int(N * self.keep_ratio)
        topk = torch.topk(scores[:, 1:], k, dim=1).indices + 1

        cls_idx = torch.zeros(B, 1, dtype=torch.long, device=x.device)
        keep_idx = torch.cat([cls_idx, topk], dim=1)

        x = torch.gather(x, 1, keep_idx.unsqueeze(-1).expand(-1, -1, C))
        x = self.base.encoder.layers(x)
        x = self.base.encoder.ln(x)

        cls = x[:, 0]
        return self.base.heads(cls)

# -------------------------
# Load Teacher
# -------------------------
teacher = vit_b_16(weights=None)
teacher.heads.head = nn.Linear(teacher.heads.head.in_features, 100)
teacher.load_state_dict(torch.load("/kaggle/input/datasets/omswarooptm/densevit-cifar-100/dense_cifar100_best.pt", map_location=device))
teacher.to(device)
teacher.eval()

print("Teacher Accuracy:", evaluate(teacher))

# -------------------------
# KD Training
# -------------------------
def train_kd(model, teacher, epochs=20, alpha=0.7, T=4.0):
    model.to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=3e-5)
    scaler = torch.amp.GradScaler("cuda")

    for epoch in range(epochs):
        model.train()
        total_loss = 0
        for x, y in train_dl:
            x, y = x.to(device), y.to(device)
            optimizer.zero_grad()

            with torch.amp.autocast("cuda"):
                out = model(x)
                ce = F.cross_entropy(out, y)

                with torch.no_grad():
                    t_out = teacher(x)

                kd = F.kl_div(
                    F.log_softmax(out/T, dim=1),
                    F.softmax(t_out/T, dim=1),
                    reduction="batchmean"
                ) * (T*T)

                loss = alpha * kd + (1 - alpha) * ce

            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            total_loss += loss.item()

        acc = evaluate(model)
        print(f"KD Epoch {epoch+1}/{epochs} - Loss: {total_loss/len(train_dl):.4f} - Val Acc: {acc:.4f}")

    return model

print("\nTraining Sparse + KD...")
student = SparseViT(copy.deepcopy(teacher), keep_ratio=0.5)
student = train_kd(student, teacher, epochs=20)
kd_acc = evaluate(student)
print("Final Sparse + KD Accuracy:", kd_acc)
torch.save(student.state_dict(), "sparse_kd_cifar100.pt")

Using device: cuda


100%|██████████| 169M/169M [00:08<00:00, 20.5MB/s] 


Teacher Accuracy: 0.8703

Training Sparse + KD...
KD Epoch 1/20 - Loss: 0.9593 - Val Acc: 0.7706
KD Epoch 2/20 - Loss: 0.5353 - Val Acc: 0.7562
KD Epoch 3/20 - Loss: 0.3833 - Val Acc: 0.7577
KD Epoch 4/20 - Loss: 0.3038 - Val Acc: 0.7756
KD Epoch 5/20 - Loss: 0.2640 - Val Acc: 0.7828
KD Epoch 6/20 - Loss: 0.2297 - Val Acc: 0.7729
KD Epoch 7/20 - Loss: 0.2134 - Val Acc: 0.7823
KD Epoch 8/20 - Loss: 0.2013 - Val Acc: 0.7708
KD Epoch 9/20 - Loss: 0.1775 - Val Acc: 0.7734
KD Epoch 10/20 - Loss: 0.1693 - Val Acc: 0.7747
KD Epoch 11/20 - Loss: 0.1640 - Val Acc: 0.7675
KD Epoch 12/20 - Loss: 0.1759 - Val Acc: 0.7633
KD Epoch 13/20 - Loss: 0.1510 - Val Acc: 0.7735
KD Epoch 14/20 - Loss: 0.1516 - Val Acc: 0.7743
KD Epoch 15/20 - Loss: 0.1387 - Val Acc: 0.7688
KD Epoch 16/20 - Loss: 0.1474 - Val Acc: 0.7652
KD Epoch 17/20 - Loss: 0.1405 - Val Acc: 0.7809
KD Epoch 18/20 - Loss: 0.1327 - Val Acc: 0.7763
KD Epoch 19/20 - Loss: 0.1290 - Val Acc: 0.7599
KD Epoch 20/20 - Loss: 0.1342 - Val Acc: 0.7728